In [24]:
# Find customers (ASN and prefixes) of DDoS scrubber NBIP-NaWaS (AS200020)for a day
import pybgpstream
import ipaddress # Used to get network mask of a prefix (IPv4 or IPv6) and its version
import pytricia  # Used to store prefix to avoid duplicate records that happens 
#because of multiple route collectors dumping a same prefix data 

stream = pybgpstream.BGPStream(
    from_time="2024-07-07 00:00:01 CET",
    until_time="2024-07-07 23:59:00 CET",
    collectors=["rrc00"],#, "rrc03", "rrc25", "route-views.amsix"],
    record_type="updates",     
        filter = "path _200020_" #Look for all the prefixes originated by AS200020
   )
stream.set_data_interface_option("broker", "cache-dir", "/home/shyam/jupy/cache")
stream.add_filter('elemtype', 'announcements')

# List of prefixes that AS200020 owns 
# Source https://stat.ripe.net/data/ris-prefixes/data.json?resource=200020&list_prefixes=1

pfx_list = ["194.62.131.0/24",
            "212.114.112.0/24",
            "2001:67c:608::/48"
            ]

pyt_v4 = pytricia.PyTricia() # For storing ipv4 prefixes
pyt_v4 = pytricia.PyTricia(128) # For storing ipv6 prefixes



# Find paths from DDoS scrubber to route collectors
for rec in stream.records():
    time = rec.time
    for elem in rec:
#         print("## elem %s" %elem)
        peer_asn = elem.peer_asn
        peer_ip = elem.peer_address
        
        pfx = elem.fields["prefix"]
        ip = ipaddress.ip_network(pfx)
        pfx_len = ip.prefixlen
        
        # Find second ASN in an AS path
        as_path = elem.fields["as-path"]
        orig = as_path.split()[-1]
        second_as = as_path.split()[-2]

        # Store origin asn and announcement time in a dictionary
        asn_time = {}
        
        # Condition that it is a scrub AS 
        # 1. AS should be the second last AS in an AS path
        # 2. Prefix length should be /24 or /48
        # See all the prefixes that has AS200020 as a second last hop
        if second_as == "200020":# and pfx_len == 24:
#             print("Prefix %s, asn %s , time %s" %(pfx, orig, time))
            asn_time["asn"] = orig
            asn_time["time"] = time
            asn_time["peer_asn"] = peer_asn
            asn_time["peer_ip"] = peer_ip

            # Store prefix, origin, timestamp, peer collector in a dictionary in a patricia tree
            pyt_v4.insert(pfx, asn_time)                   
#         print(len(pyt_v4))   
print("Completed")
        

Completed


In [25]:
len(pyt_v4)

32

In [26]:
# Store pytricia objects into a csv file
import pandas as pd
data_list = [{'prefix': prefix, 'asn': pyt_v4[prefix]['asn'], 'announced_time': pyt_v4[prefix]['time'], 'peer_asn': pyt_v4[prefix]['peer_asn'], 'peer_ip': pyt_v4[prefix]['peer_ip']} for prefix in pyt_v4]

# transformed_data = [{'prefix': item[0], 'asn': item[1]['asn']} for item in data]


# df = pd.DataFrame(data_list, columns=['Prefix', 'ASN', 'Time', 'peer ASN', 'peer IP'], index=False)
df = pd.DataFrame(data_list)
df.to_csv('/home/shyam/jupy/ddos_scrubber/data/as200020_07July_rrc00_pflen_any.csv', index = False)

In [69]:
# Get number of unique ASNs for each days
df1 = pd.read_csv('/home/shyam/jupy/ddos_scrubber/data/as200020_07July_rrc00_pflen_any.csv')
asn_df1 = df1.asn.unique()
asn_df1.sort()
asn_df1


array([ 8315,  9031, 12414, 12859, 15703, 20847, 24586, 29396, 31673,
       34373, 39591, 39647, 41887, 49685, 50266, 51758, 60294])

In [42]:
# Get number of unique prefixes for each days
df1 = pd.read_csv('/home/shyam/jupy/ddos_scrubber/data/as200020_01July_rrc00_pflen_any.csv')
prefix_df1 = df1.prefix
prefix_df1

0        2001:7be::/48
1     2001:4018:3::/48
2       62.165.82.0/24
3      77.109.108.0/24
4       83.98.201.0/24
5      85.146.160.0/24
6      85.158.160.0/24
7      85.234.195.0/24
8        86.48.69.0/24
9       89.255.17.0/24
10      90.145.87.0/24
11      93.92.103.0/24
12     94.105.121.0/24
13     95.215.185.0/24
14     95.215.189.0/24
15      145.131.7.0/24
16      149.146.3.0/24
17     185.12.147.0/24
18     185.158.41.0/24
19    185.220.110.0/24
20      188.89.97.0/24
21      192.54.67.0/24
22     212.83.192.0/24
23     212.83.206.0/24
24     213.207.64.0/24
25    213.211.131.0/24
26    213.211.155.0/24
27    213.211.172.0/24
28    213.211.176.0/24
29    213.219.129.0/24
30    213.219.171.0/24
31    213.219.174.0/24
32    213.219.183.0/24
33    213.219.187.0/24
34     217.21.248.0/24
Name: prefix, dtype: object

In [ ]:
# Check withdrawal time of a prefix 
import pybgpstream
import ipaddress # Used to get network mask of a prefix (IPv4 or IPv6) and its version
import pytricia  # Used to store prefix to avoid duplicate records that happens 
#because of multiple route collectors dumping a same prefix data 

stream = pybgpstream.BGPStream(
    from_time="2024-07-01 00:00:01 CET",
    until_time="2024-07-01 23:59:00 CET",    collectors=["rrc00"],#, "rrc03", "rrc25", "route-views.amsix"],
    record_type="updates",     
   )
stream.set_data_interface_option("broker", "cache-dir", "/home/shyam/jupy/cache")
stream.add_filter('elemtype', 'withdrawals')
stream.add_filter('prefix', '83.98.201.0/24')

collector_peer = "34800" # Change here peer ASN as received from the announcement type

# Find paths from DDoS scrubber to route collectors
for rec in stream.records():
    for elem in rec:
        if elem.peer_asn = collector_peer:
            break
    time = rec.time
print(time)
# print("##time")           


In [ ]:
# TODO: Characterizing the DDoS scrbber customer
# 1. Check withdrawal time of a prefix
# 2. Find organization. Check if a same organization owns those customers. For example, Signet B.V. owns 9 ASNs 
# 3. Find AS type
import pybgpstream
import ipaddress # Used to get network mask of a prefix (IPv4 or IPv6) and its version

stream = pybgpstream.BGPStream(
#     from_time="2024-07-01 00:00:01 CET",
    from_time = "2024-07-01 00:31:00 CET",
    until_time="2024-07-05 23:59:00 CET",    
    collectors=["rrc00"],#, "rrc03", "rrc25", "route-views.amsix"],
    record_type="updates",  
    filter = "path 50266$" #Look for all the prefixes originated by AS200020

   )
stream.set_data_interface_option("broker", "cache-dir", "/home/shyam/jupy/cache")
stream.add_filter('prefix-any', '188.89.97.0/24')
stream.add_filter('elemtype', 'withdrawals')

for rec in stream.records():
    for elem in rec:
        peer_asn = elem.peer_asn
        pfx = elem.fields["prefix"]
#         print("Prefix is %s" %pfx)
        if pfx != "0.0.0.0/0":# and peer_asn = "49673":
            print(elem)


In [ ]:
import pybgpstream
# Sample program to see announcement from DDoS scrubber
# Find paths from AS57724 (DDoS Guard)
stream = pybgpstream.BGPStream(
    from_time="2020-07-01 00:00:01 CET",
    until_time="2024-07-03 23:59:11 CET",
#     collectors=["rrc00", "rrc03", "rrc25", "route-views.amsix"],
    record_type="ribs",     
        filter = 'path _57724' #Look for all the prefixes originated by AS200020
   )
stream.set_data_interface_option("broker", "cache-dir", "/home/shyam/jupy/cache")

# List of prefixes that 
pfx_list = ["45.10.240.0/24", "45.10.243.0/24", "45.132.16.0/24", "45.155.60.0/24", "77.220.207.0/24", 
            "91.215.40.0/24", "91.215.41.0/24", "91.215.42.0/24", "91.215.43.0/24","95.129.232.0/24", 
            "95.129.233.0/24","95.129.234.0/24","176.57.64.0/23", "185.9.185.0/24", "185.129.100.0/24",
            "185.129.101.0/24","185.129.102.0/24",
            "185.129.103.0/24",
            "185.149.120.0/24",
            "185.178.208.0/24",
            "185.178.209.0/24",
            "185.178.210.0/24",
            "185.215.4.0/24",
            "185.223.92.0/24",
            "195.216.243.0/24",
            "217.114.42.0/24",
            "2a0a:4180::/48", ]

# Find paths from DDoS scrubber to route collectors
for rec in stream.records():
    for elem in rec:
        pfx = elem.fields["prefix"]
        as_path = elem.fields["as-path"]
        orig = as_path.split()[-1]
#         print("Origin is %s and prefix is %s" %(orig, pfx))
        if orig == "57724" and pfx not in (pfx_list):
            print(elem)
        # Find prefix except it owns 


In [ ]:
import pybgpstream
import ipaddress # Used to get network mask of a prefix (IPv4 or IPv6) and its version
import pytricia  # Used to store prefix to avoid duplicate records that happens 
#because of multiple route collectors dumping a same prefix data 

stream = pybgpstream.BGPStream(
    from_time="2024-07-01 00:00:01 CET",
    until_time="2024-07-01 23:59:00 CET",
    collectors=["rrc00"],#, "rrc03", "rrc25", "route-views.amsix"],
    record_type="ribs",     
        filter = "path 20847$" #Look for all the prefixes originated by AS200020
   )
stream.set_data_interface_option("broker", "cache-dir", "/home/shyam/jupy/cache")
#stream.add_filter('elemtype', 'announcements')
for rec in stream.records():
    for elem in rec:
        print(elem)

In [10]:
#------------------------STEP #1---------------------------------------------------------------------------
# For Cloudflare AS13335, find customers (ASN and prefixes) of DDoS scrubber NBIP-NaWaS (AS200020)for a day
import pybgpstream
import ipaddress # Used to get network mask of a prefix (IPv4 or IPv6) and its version
import pytricia  # Used to store prefix to avoid duplicate records that happens 
#because of multiple route collectors dumping a same prefix data 

stream = pybgpstream.BGPStream(
    from_time="2024-07-01 00:00:00 CET",
    until_time="2024-07-01 23:59:00 CET",
#     collectors=["rrc00"],#, "rrc03", "rrc25", "route-views.amsix"],
    record_type="updates",     
        filter = "path _13335_" #Look for all the prefixes where AS1335 comes in an AS path
   )
stream.set_data_interface_option("broker", "cache-dir", "/home/shyam/jupy/cache")
stream.add_filter('elemtype', 'announcements')

pyt_v4 = pytricia.PyTricia() # For storing ipv4 prefixes
pyt_v4 = pytricia.PyTricia(128) # For storing ipv6 prefixes



# Find paths from DDoS scrubber to route collectors
for rec in stream.records():
    time = rec.time
    for elem in rec:
#         print("## elem %s" %elem)
        peer_asn = elem.peer_asn
        peer_ip = elem.peer_address
        
        pfx = elem.fields["prefix"]
        ip = ipaddress.ip_network(pfx)
        pfx_len = ip.prefixlen
        
        # Find second ASN in an AS path
        as_path = elem.fields["as-path"]
        
        # Convert as_path into list
        as_path_list = as_path.split()
        
        orig = as_path_list[-1]
        
        # Discard different forms of origins example AS-Set, confederation set/sequence. 
        # Take only a single AS origin which is common for a scrubbing activity
        if validate_origin(orig) and len(as_path_list) > 1:
            second_as = as_path.split()[-2]

            # Store prefix details such as origin asn, announcement time, collector asn and peer IP in a dictionary
            prefix_details = {}

            # Condition that it is a scrub AS 
            # 1. AS should be the second last AS in an AS path
            # 2. Prefix length should be /24 or /48
            if second_as == "13335" and pfx_len == 24:
    #             print("Prefix %s, asn %s , time %s" %(pfx, orig, time))
                prefix_details["asn"] = orig# Store pytricia objects into a csv file
import pandas as pd
data_list = [{'prefix': prefix, 'asn': pyt_v4[prefix]['asn'], 'announced_time': pyt_v4[prefix]['time'], 'peer_asn': pyt_v4[prefix]['peer_asn'], 'peer_ip': pyt_v4[prefix]['peer_ip']} for prefix in pyt_v4]

# transformed_data = [{'prefix': item[0], 'asn': item[1]['asn']} for item in data]


# df = pd.DataFrame(data_list, columns=['Prefix', 'ASN', 'Time', 'peer ASN', 'peer IP'], index=False)
df = pd.DataFrame(data_list)
                prefix_details["time"] = time
                prefix_details["peer_asn"] = peer_asn
                prefix_details["peer_ip"] = peer_ip

                # Store prefix, origin, timestamp, peer collector in a dictionary in a patricia tree
                pyt_v4.insert(pfx, prefix_details)                   
#         print(len(pyt_v4))   
print("Completed")
        

1721124352 HTTP ERROR: Transferred a partial file (18)
1721124367 HTTP ERROR: Couldn't connect to server (7)
2024-07-16 12:06:07 8375: bs_transport_cache.c:244: ERROR: ERROR: Could not open http://archive.routeviews.org/route-views.fortaleza/bgpdata/2024.07/UPDATES/updates.20240701.1545.bz2 for reading
2024-07-16 12:06:07 8375: bgpstream_transport.c:97: ERROR: Could not open resource (http://archive.routeviews.org/route-views.fortaleza/bgpdata/2024.07/UPDATES/updates.20240701.1545.bz2)
2024-07-16 12:06:07 8375: bgpstream_reader.c:169: WARNING: Could not open (http://archive.routeviews.org/route-views.fortaleza/bgpdata/2024.07/UPDATES/updates.20240701.1545.bz2). Attempt 1 of 5
1721124367 HTTP ERROR: Couldn't connect to server (7)
2024-07-16 12:06:07 8375: bs_transport_cache.c:244: ERROR: ERROR: Could not open http://archive.routeviews.org/route-views.nwax/bgpdata/2024.07/UPDATES/updates.20240701.1600.bz2 for reading
2024-07-16 12:06:07 8375: bgpstream_transport.c:97: ERROR: Could not op

KeyboardInterrupt: 

In [11]:
len(pyt_v4)

0

In [8]:
# STEP 1: Helper function to filter AS paths to only include those of the form "100 200 300" 
# and with at least two elements.
# Returns bool: True if the AS path is valid, False otherwise.

import re

def validate_origin(as_path):
    valid = False
    # Check if the path matches the pattern: only space-separated numbers
    if re.fullmatch(r'(\d+\s)+\d+', as_path):
        # Split the path by spaces to count the elements
        elements = as_path.split()
        if len(elements) >= 2:
            valid = True
    return valid
    
# Example usage
as_path = "300"
"""
,

"{36040,38266,45271,55410}",
    "[12345,6789]",
    "(12345 6789)",
    "100",
    "200 300",
    "abc def ghi",
    "400 500 600 700"
filtered_paths = validate_as_path(as_path)
print(filtered_paths)
"""




'\n,\n\n"{36040,38266,45271,55410}",\n    "[12345,6789]",\n    "(12345 6789)",\n    "100",\n    "200 300",\n    "abc def ghi",\n    "400 500 600 700"\nfiltered_paths = validate_as_path(as_path)\nprint(filtered_paths)\n'

In [ ]:
# Store pytricia objects into a csv file
import pandas as pd
data_list = [{'prefix': prefix, 'asn': pyt_v4[prefix]['asn'], 'announced_time': pyt_v4[prefix]['time'], 'peer_asn': pyt_v4[prefix]['peer_asn'], 'peer_ip': pyt_v4[prefix]['peer_ip']} for prefix in pyt_v4]

# transformed_data = [{'prefix': item[0], 'asn': item[1]['asn']} for item in data]


# df = pd.DataFrame(data_list, columns=['Prefix', 'ASN', 'Time', 'peer ASN', 'peer IP'], index=False)
df = pd.DataFrame(data_list)

df.to_csv('/home/shyam/jupy/ddos_scrubber/data/as13335_01July.csv', index=False)

In [1]:
#----------------------STEP #2-----------------------------
# 1. Check if the prefix is withdrawn by the same peer ASN
# 2. Find organization. Check if a same organization owns those customers. For example, Signet B.V. owns 9 ASNs 
# 3. Find AS type
"""
# For each prefix in that list check AS path of its less specific announcement 
for index, row in df.iterrows():
    pfx = row["prefix"]
    org = row["asn"]
    peer = row["peer_asn"]
          
"""
# For now checking for an individual prefix manually                
import pybgpstream
import ipaddress # Used to get network mask of a prefix (IPv4 or IPv6) and its version
# For each prefix in that list check AS path of its less specific announcement 
# pfx = "2.59.52.0/24"
pfx = "193.58.155.0/24"
peer = "55720" # Collector peer ASN
org = "2047" # Origin
time = "1719866886" # Announced time


stream = pybgpstream.BGPStream(
    from_time = "2024-07-01 00:00:00",
    until_time="2024-07-05 23:59:00",    
    collectors=["rrc00"],#, "rrc03", "rrc25", "route-views.amsix"],
    record_type="updates" 
   )
stream.set_data_interface_option("broker", "cache-dir", "/home/shyam/jupy/cache")
stream.add_filter('prefix', pfx)
stream.add_filter('elemtype', 'withdrawals')

find = False # Boolean to indicate if record is found or not
elem_val = None # To store elem 

# Flag to indicate if we should break the outer loop
break_outer_loop = False


for rec in stream.records():
    for elem in rec:
        peer_asn = str(elem.peer_asn)  # Ensure peer_asn is compared as a string
        prefix = elem.fields["prefix"]
        print(elem)
        """
        
        if peer_asn == peer:
            find = True
            time = rec.time
            elem_val = elem
            break_outer_loop = True
            break
        
    if break_outer_loop:
        break

if find:
    print("Withdrawal time %s and element  %s" %(time,elem_val))
    
    # GOTO Step #3 to see AS path pattern of its less specific prefix
    
else:
    print("Not withdrawn prefix ==> always-ON scrubbing")
    # Go to Step 5 to check if other AS path exists without the presence of scrubber ASN
    
# TODO: Ignore the prefixes whose announcement and withdrawal time has less than 2 mins of duration     
      """
  
    

update|W|1719839901.000000|ris|rrc00|None|None|34800|193.163.86.231|193.58.155.0/24|None|None|None|None|None


KeyboardInterrupt: 

In [81]:
#-------STEP #3: Check announcement pattern of a prefix not withdrawn for more than 7 days-------------------
# 1. Check if a less specific announcement exists after the announcement time of the specific prefix
# 2. Look for the less specific announcements of the prefix for the next 7 days after announcement time as seen by 
# the same collector peer
# if it exist and does not have a scrubber ASN in its AS path, it is on-demand scrubbing. 

import pybgpstream
import time

prefix = "213.211.176.0/24"
org = "9031" # Origin ASN
peer = "49673" # Collector peer ASN
scrubber = "200020" # Scrubber ASN

announced_time_unix = int("1719832296") # To convert string format stored in CSV file to int 
announced_time = time.strftime('%Y-%m-%d %H:%M:%S', time.gmtime(announced_time_unix))

stream = pybgpstream.BGPStream(
#     from_time="2024-07-01 00:00:01 CET",
    from_time = announced_time,
    until_time="2024-07-08 23:59:00 CET",    
    collectors=["rrc00"],#, "rrc03", "rrc25", "route-views.amsix"],
    record_type="updates",  
    filter = "path "+org+"$" #Look for all the prefixes originated by AS200020

   )
stream.set_data_interface_option("broker", "cache-dir", "/home/shyam/jupy/cache")
stream.add_filter('prefix-less', prefix)
stream.add_filter('elemtype', 'announcements')


find = False # Boolean to indicate if record is found or not
elem_val = None # To store elem 

# Flag to indicate if we should break the outer loop
break_outer_loop = False


for rec in stream.records():
    
    for elem in rec:
        pfx = elem.fields["prefix"]
        as_path = elem.fields["as-path"].split()
        
        peer_asn = str(elem.peer_asn)  # Ensure peer_asn is compared as a string

        if pfx != "0.0.0.0/0" and is_less_specific(pfx, prefix) and peer_asn == peer and scrubber not in as_path:
            elem_val = elem
            break_outer_loop = True
            find = True
            break
    if break_outer_loop:
        break

if find:
    print("On-demand scrubbing for %s" %(elem_val))
else:
    print("No scrubbing")

Element  update|A|1719916569.000000|ris|rrc00|None|None|49673|94.247.111.254|213.211.176.0/21|94.247.111.254|49673 48858 9031|9031:11001 9031:10001 9031:10101 48858:1399|None|None


In [88]:
#---------------------------STEP #4: Check announcement pattern of the withdrawn prefix--------------------------------------------------------------------------
import pybgpstream
import time

prefix = "62.165.82.0/24"
peer = "202365"
org = "20847"
announced_time_unix = "1719874184" # Announced time
scrubber = "200020" # Scrubber ASN

announced_time_unix = int(announced_time_unix) # To convert string format stored in CSV file to int 
announced_time = time.strftime('%Y-%m-%d %H:%M:%S', time.gmtime(announced_time_unix))

stream = pybgpstream.BGPStream(
#     from_time="2024-07-01 00:00:01 CET",
    from_time = announced_time,
    until_time="2024-07-02 23:59:00 CET",    
    collectors=["rrc00"],#, "rrc03", "rrc25", "route-views.amsix"],
    record_type="updates",  
    filter = "path "+org+"$" #Look for all the prefixes originated by AS200020

   )
stream.set_data_interface_option("broker", "cache-dir", "/home/shyam/jupy/cache")
stream.add_filter('prefix', prefix)
stream.add_filter('elemtype', 'announcements')


find = False # Boolean to indicate if record is found or not
elem_val = None # To store elem 

# Flag to indicate if we should break the outer loop
break_outer_loop = False


for rec in stream.records():
    
    for elem in rec:
        pfx = elem.fields["prefix"]
        as_path = elem.fields["as-path"].split()
        
        peer_asn = str(elem.peer_asn)  # Ensure peer_asn is compared as a string

        if peer_asn == peer and scrubber not in as_path:
            elem_val = elem
            break_outer_loop = True
            find = True
            break
    if break_outer_loop:
        break

if find:
    print(f"Always-ON scrubbing for {elem_val} with path prepend")
else:
    print(f"Always-ON scrubbing for {elem_val} without path prepend")

No scrubbing


In [72]:
prefix = "192.168.1.0/24"
superprefix = "192.168.0.0/22"
def is_less_specific(superprefix, prefix):
    """
    Check if the given prefix is less specific than the supernet.
    """
    import ipaddress

    prefix_net = ipaddress.ip_network(prefix)
    supernet_net = ipaddress.ip_network(superprefix)
    return supernet_net.prefixlen < prefix_net.prefixlen 
#     return supernet_net.subnet_of(prefix_net) and supernet_net.prefixlen < prefix_net.prefixlen 

In [3]:
# Step # 5. For prefixes not withdrawan, check if there exists some AS paths without having the scrubber ASNs

import pybgpstream
from datetime import datetime

prefix = "131.215.240.0/24"
peer = "55720" # Collector peer ASN
org = "31" # Origin
time = "1719866886" # Announced time
scrubber = "13335" # Scrubber ASN

announced_time_unix = int(time) # To convert string format stored in CSV file to int 
announced_time = datetime.utcfromtimestamp(announced_time_unix).strftime('%Y-%m-%d %H:%M:%S')

stream = pybgpstream.BGPStream(
#     from_time="2024-07-01 00:00:01 CET",
    from_time = announced_time,
    until_time="2024-07-08 23:59:00 CET",    
    collectors=["rrc00"],#, "rrc03", "rrc25", "route-views.amsix"],
    record_type="updates",  
    filter = "path "+org+"$" #Look for all the prefixes originated by origin ASN

   )
stream.set_data_interface_option("broker", "cache-dir", "/home/shyam/jupy/cache")
stream.add_filter('prefix-less', prefix)
stream.add_filter('elemtype', 'announcements')


find = False # Boolean to indicate if record is found or not
elem_val = None # To store elem 

# Flag to indicate if we should break the outer loop
break_outer_loop = False


for rec in stream.records():
    
    for elem in rec:
        pfx = elem.fields["prefix"]
        as_path = elem.fields["as-path"].split()
        
        peer_asn = str(elem.peer_asn)  # Ensure peer_asn is compared as a string
        
#         print("Elem is %s" %elem)
        if pfx != "0.0.0.0/0" and scrubber not in as_path:
            elem_val = elem
            break_outer_loop = True
            find = True
            break
    if break_outer_loop:
        break

if find:
    print("###Less specific announcement exists without scrubber asn => On-demand scrubbing for %s" %(elem_val))
else:
    print("No scrubbing")

###Less specific announcement exists without scrubber asn => On-demand scrubbing for update|A|1719879896.000000|ris|rrc00|None|None|202365|194.50.19.4|131.215.0.0/16|194.50.19.4|202365 6939 226 226 31|0:13335 0:16509 0:15169 0:16265 0:12989 0:12876 0:22822 0:16276 0:32590 0:20940 0:49029 0:714 0:2906 0:48641 0:6939 0:15133|None|None
